# Example of DFT calculation with Ecut and K-point convergence
# NSCI0032/2025-2026 - Term 1

## Import various libraries 

In [1]:


import os, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_PLACES"] = "cores"
os.environ["OMP_PROC_BIND"] = "close"


In [2]:
a0=3.615 / 0.529 
a1=a0*1.1
print(a0, a1)

6.833648393194707 7.517013232514178


# The Pseudo Potentials, and general setup

In [3]:
pp_dir  = "/home/jovyan/espresso/pseudo"
pp_file = "Cu.pbe-dn-kjpaw_psl.1.0.0.UPF"
pp_path = f"{pp_dir}/{pp_file}"

os.makedirs(pp_dir, exist_ok=True)

if not os.path.exists(pp_path):
    !wget -q https://pseudopotentials.quantum-espresso.org/upf_files/{pp_file} -O {pp_path}

In [4]:
ry_to_mev = 13605.698  # unit conversion

# QE Input file template

In [5]:
template = f"""&control
    calculation='scf',
    prefix='cu',
    outdir='./tmp',
    pseudo_dir='{pp_dir}'
    verbosity='high'
/
&system
    ibrav=2, celldm(1)={{a0}},
    nat=1, ntyp=1,
    ecutwfc={{ecut}},
    occupations='smearing',
    smearing='mv',
    degauss=0.02
/
&electrons
    conv_thr={{n}}
/
ATOMIC_SPECIES
 Cu 63.546 {pp_file}
ATOMIC_POSITIONS crystal
 Cu 0.0 0.0 0.0
K_POINTS automatic
 {{k}} {{k}} {{k}} 1 1 1
"""

# K point sampling and Convergence

In [6]:
k = 8
ecut = 40
os.makedirs("saves", exist_ok=True)
na_list = [ (1e-2, a0) , (1e-8, a0), (1e-2, a0*1.1) , (1e-8, a0*1.1)] 
for i, (n, a) in enumerate(na_list):
    print (i, n, a)
ener_run = {}
time_run = {}

0 0.01 6.833648393194707
1 1e-08 6.833648393194707
2 0.01 7.517013232514178
3 1e-08 7.517013232514178


In [7]:
for i, (n, a) in enumerate(na_list):
    infile  = f"pw_{i}.in"
    outfile = f"pw_{i}.out"
    print(i)

    with open(infile, "w") as f:
        f.write(template.format(k=k, ecut=ecut, n=n, a0=a))

    t0 = time.time()
    !pw.x < {infile} > {outfile}
    t1 = time.time()

    with open(outfile) as f:
        txt = f.read()

    m = re.findall(r"!\s+total energy\s*=\s*([-\d\.E+]+)", txt)
    ener_run[i] = float(m[-1])
    time_run[i] = t1 - t0

0
1
2
3


In [8]:
print(ener_run, time_run)

{0: -213.09273093, 1: -213.09282325, 2: -213.07066162, 3: -213.0706779} {0: 5.36076283454895, 1: 6.680438280105591, 2: 8.388267040252686, 3: 12.504407167434692}


In [9]:
(ener_run[2]-ener_run[0])*ry_to_mev

300.268366928078

In [10]:
(time_run[2]+time_run[0])/2

6.874514937400818

In [11]:
(ener_run[3]-ener_run[1])*ry_to_mev

301.30294420452594

In [12]:
(time_run[3]+time_run[1])/2

9.592422723770142